In [3]:
import json
import glob
import os

# ── Load all 4 JSON files from the folder ────────────────────────────────────
folder = '../data/CVR_API/'
json_files = sorted([f for f in glob.glob(os.path.join(folder, '*.json')) if 'cvr_production_units_final' not in os.path.basename(f)])
# Exclue 

all_records   = []
vat_seen      = {}   # vat -> list of filenames it appears in

for filepath in json_files:
    filename = os.path.basename(filepath)
    with open(filepath, 'r', encoding='utf-8') as f:
        data = json.load(f)

    print(f"  {filename}: {len(data)} records")
    all_records.extend(data)

    for record in data:
        vat = record.get('vat')
        name = record.get('name', '')
        if vat is not None and name not in ("Jonathan Heavens Wolt Denmark", "Edifice Housing and Projects A/S"):
            vat_seen.setdefault(vat, []).append(filename)
            
# ── Totals ────────────────────────────────────────────────────────────────────
total_records   = len(all_records)
total_vat       = len(vat_seen)

# VATs that appear in exactly one file (non-repeating / unique)
unique_vats     = {vat: files for vat, files in vat_seen.items() if len(files) == 1}
# VATs that appear more than once (duplicates)
duplicate_vats  = {vat: files for vat, files in vat_seen.items() if len(files) > 1}

print(f"\n── Summary ──────────────────────────────────────────────")
print(f"  JSON files found:                  {len(json_files)}")
print(f"  Total records (all 4 files):       {total_records}")
print(f"  Total distinct VAT values:         {total_vat}")
print(f"  Non-repeating VATs (appear once):  {len(unique_vats)}")
print(f"  Duplicate VATs (appear >1 time):   {len(duplicate_vats)}")

if duplicate_vats:
    print(f"\n── Duplicate VATs ───────────────────────────────────────")
    for vat, files in duplicate_vats.items():
        print(f"  VAT {vat} → found in: {', '.join(files)}")

  2_cvr_production_units.json: 33 records
  Andrea_cvr_production_units.json: 50 records
  Uliyan_cvr_production_units.json: 50 records
  cvr_production_units.json: 36 records

── Summary ──────────────────────────────────────────────
  JSON files found:                  4
  Total records (all 4 files):       169
  Total distinct VAT values:         157
  Non-repeating VATs (appear once):  149
  Duplicate VATs (appear >1 time):   8

── Duplicate VATs ───────────────────────────────────────
  VAT 64806815 → found in: 2_cvr_production_units.json, 2_cvr_production_units.json
  VAT 35954716 → found in: 2_cvr_production_units.json, Andrea_cvr_production_units.json
  VAT 12626509 → found in: 2_cvr_production_units.json, cvr_production_units.json
  VAT 40075291 → found in: 2_cvr_production_units.json, 2_cvr_production_units.json, Andrea_cvr_production_units.json, Andrea_cvr_production_units.json
  VAT 24260666 → found in: 2_cvr_production_units.json, 2_cvr_production_units.json
  VAT 47458714

In [5]:
import pandas as pd


# ── Load cvr_production_units_final.json ─────────────────────────────────────
with open('../data/CVR_API/cvr_production_units_final.json', 'r', encoding='utf-8') as f:
    results = json.load(f)

# ── Build production units DataFrame ─────────────────────────────────────────
data_rows  = []
index_list = []
keys    = ['name', 'address', 'zipcode', 'city', 'startdate', 'enddate', 'employees']
columns = ['UnitPno', 'CompanyVat', 'CompanyName', 'UnitName', 'UnitAddress', 'UnitZipcode', 'UnitCity', 'UnitStartdate', 'UnitEnddate', 'UnitEmployees']

for res in results:
    for unit in res.get('productionunits', []):
        pno    = unit.get('pno', None)
        values = [pno , res['vat'], res['name']] + [unit.get(key, None) for key in keys]
        data_rows.append(values)
        index_list.append(pno)

df_units = pd.DataFrame(data_rows, columns=columns)
# df_units.index.name = 'pno'

print(f"Total production units: {len(df_units)}")


Total production units: 28071


In [6]:
# ── Parse UnitEnddate and filter ─────────────────────────────────────────────
df_units['UnitEnddate'] = pd.to_datetime(
    df_units['UnitEnddate'].str.replace(' - ', '/', regex=False),  # "15/06 - 1991" → "15/06/1991"
    format='%d/%m/%Y',
    errors='coerce'                                                 # unparseable → NaT (still open)
)

cutoff = pd.Timestamp('2008-01-01')

# Keep rows where UnitEnddate is NaT (still open) OR closed on/after 2008
df_units = df_units[df_units['UnitEnddate'].isna() | (df_units['UnitEnddate'] >= cutoff)]

print(f"Units after purge (operative from 2008 onwards): {len(df_units)}")

Units after purge (operative from 2008 onwards): 25605


In [12]:
import time
import requests
# ── Geocode addresses using DAWA (Danish Address Web API) ─────────────────────
# Free, no API key needed, covers all Danish addresses
DAWA_URL = "https://api.dataforsyningen.dk/adresser"

def geocode_danish_address(address, zipcode, city):
    try:
        params = {
            'q'       : f"{address}, {zipcode} {city}",
            'per_side': 1,
            'struktur': 'nestet'
        }
        resp = requests.get(DAWA_URL, params=params, timeout=10)
        resp.raise_for_status()
        hits = resp.json()
        if hits:
            hit          = hits[0]
            coords       = hit['adgangsadresse']['vejpunkt']['koordinater']
            kommune      = hit['adgangsadresse']['kommune']
            region       = hit['adgangsadresse']['region']
            kommune_name = kommune.get('navn', None)
            kommune_code = kommune.get('kode', None)
            region_name  = region.get('navn', None)
            region_code  = region.get('kode', None)
            return coords[1], coords[0], kommune_name, kommune_code, region_name, region_code
    except Exception as e:
        print(f"  ✗ Failed [{address}, {zipcode}]: {e}")
    return None, None, None, None, None, None

lats, lons, kommune_names, kommune_codes, region_names, region_codes = [], [], [], [], [], []

for i, (_, row) in enumerate(df_units.iterrows()):
    lat, lon, k_name, k_code, r_name, r_code = geocode_danish_address(row['UnitAddress'], row['UnitZipcode'], row['UnitCity'])
    lats.append(lat)
    lons.append(lon)
    kommune_names.append(k_name)
    kommune_codes.append(k_code)
    region_names.append(r_name)
    region_codes.append(r_code)

    if (i + 1) % 50 == 0:
        print(f"  Geocoded {i + 1}/{len(df_units)}...")
    time.sleep(0.05)

df_units['Lat']         = lats
df_units['Lon']         = lons
df_units['KommuneName'] = kommune_names
df_units['KommuneCode'] = kommune_codes
df_units['RegionName']  = region_names
df_units['RegionCode']  = region_codes

failed = df_units['Lat'].isna().sum()
print(f"\n✓ Geocoded {len(df_units) - failed}/{len(df_units)} units successfully")
print(f"✗ Failed to geocode: {failed} units")

# ── Save ──────────────────────────────────────────────────────────────────────
df_units.to_csv('../data/CVR_API/cvr_production_units_geocoded.csv', encoding='utf-8')
print("Saved → cvr_production_units_geocoded.csv")

  Geocoded 50/25605...
  Geocoded 100/25605...
  Geocoded 150/25605...
  Geocoded 200/25605...
  Geocoded 250/25605...
  Geocoded 300/25605...
  Geocoded 350/25605...
  Geocoded 400/25605...
  Geocoded 450/25605...
  Geocoded 500/25605...
  Geocoded 550/25605...
  Geocoded 600/25605...
  Geocoded 650/25605...
  Geocoded 700/25605...
  Geocoded 750/25605...
  Geocoded 800/25605...
  Geocoded 850/25605...
  Geocoded 900/25605...
  Geocoded 950/25605...
  Geocoded 1000/25605...
  Geocoded 1050/25605...
  Geocoded 1100/25605...
  Geocoded 1150/25605...
  Geocoded 1200/25605...
  Geocoded 1250/25605...
  Geocoded 1300/25605...
  Geocoded 1350/25605...
  Geocoded 1400/25605...
  Geocoded 1450/25605...
  Geocoded 1500/25605...
  Geocoded 1550/25605...
  Geocoded 1600/25605...
  Geocoded 1650/25605...
  Geocoded 1700/25605...
  Geocoded 1750/25605...
  Geocoded 1800/25605...
  Geocoded 1850/25605...
  Geocoded 1900/25605...
  Geocoded 1950/25605...
  Geocoded 2000/25605...
  Geocoded 2050/2560